# Lab 2 — Elbow Method

**Day 05 · Unsupervised Learning · Cisco AI/ML Training**

---

## Learning objectives

1. Sweep k ∈ {2, …, 8} and record **inertia** for each K-Means fit.
2. Plot the **elbow curve** (k vs inertia).
3. Suggest k by the **largest inertia drop** between consecutive values.
4. Compare suggested k to Lab 1 baseline (k = 4).

> **Checkpoints:** suggested k = **3** · inertia table for k = **2** through **8**

**Companion script:** `../scripts/lab02_elbow_method.py`

## The elbow method

Inertia always **decreases** as k increases (more centroids → tighter clusters). The **elbow** is where returns diminish.

| k | Trade-off |
|---|----------|
| **Small** | Broader segments; higher inertia |
| **Elbow** | Best balance of simplicity vs fit |
| **Large** | Low inertia but many tiny segments (over-segmentation) |

This lab uses the same **25** NYSE symbol features as Lab 1. The script picks k with the **largest single-step inertia drop** — a simple classroom heuristic (Day 4 Lab 3 used a similar sweep for KNN).

---

## 1. Load symbol features (same as Lab 1)

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-05":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "nyse" / "nyse_stocks.csv").is_file():
            GH_ROOT = parent
            break

FEATURE_COLUMNS = ["avg_close", "volatility", "avg_volume", "avg_range"]

nyse = pd.read_csv(GH_ROOT / "data" / "nyse" / "nyse_stocks.csv", parse_dates=["date"])
nyse["range"] = nyse["high"] - nyse["low"]
features = (
    nyse.groupby("symbol")
    .agg(
        avg_close=("close", "mean"),
        volatility=("close", "std"),
        avg_volume=("volume", "mean"),
        avg_range=("range", "mean"),
    )
    .reset_index()
)
features["volatility"] = features["volatility"].fillna(0.0)

X_scaled = StandardScaler().fit_transform(features[FEATURE_COLUMNS])
print(f"symbols: {len(features)}")

---

## 2. Sweep k and record inertia

In [ ]:
k_range = range(2, 9)
inertias: list[tuple[int, float]] = []

for k in k_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append((k, model.inertia_))

drops = [
    (inertias[i][0], inertias[i - 1][1] - inertias[i][1])
    for i in range(1, len(inertias))
]
suggested_k = max(drops, key=lambda item: item[1])[0]

print("Lab 2 — Elbow method")
print("k\tinertia")
for k, inertia in inertias:
    print(f"{k}\t{inertia:.4f}")
print(f"suggested k (largest inertia drop): {suggested_k}")

---

## 3. Results table with inertia drops

In [ ]:
results_df = pd.DataFrame(inertias, columns=["k", "inertia"])
results_df["inertia_drop"] = results_df["inertia"].diff(-1) * -1
results_df.loc[results_df.index[-1], "inertia_drop"] = pd.NA
results_df["suggested"] = results_df["k"] == suggested_k
display(results_df.round(4))

Largest drop is **2 → 3** (~16.3) vs **3 → 4** (~11.3) — hence suggested k = **3**.

---

## 4. Plot k vs inertia (elbow curve)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.lineplot(data=results_df, x="k", y="inertia", marker="o", ax=ax, color="steelblue")
ax.scatter([suggested_k], [results_df.loc[results_df["k"] == suggested_k, "inertia"].iloc[0]],
           color="crimson", s=120, zorder=5, label=f"suggested k = {suggested_k}")
ax.axvline(4, color="gray", linestyle="--", alpha=0.5, label="Lab 1 baseline k=4")
ax.set_xlabel("k (clusters)")
ax.set_ylabel("inertia")
ax.set_title("Elbow method: inertia vs k")
ax.set_xticks(list(k_range))
ax.legend()
plt.tight_layout()
plt.show()

---

## 5. Compare to Lab 1 (k = 4)

In [ ]:
inertia_k3 = next(v for k, v in inertias if k == 3)
inertia_k4 = next(v for k, v in inertias if k == 4)

compare = pd.DataFrame({
    "setting": ["Lab 2 elbow (k=3)", "Lab 1 baseline (k=4)"],
    "k": [3, 4],
    "inertia": [inertia_k3, inertia_k4],
})
display(compare.round(4))
print(f"inertia reduction 4→3: {inertia_k3 - inertia_k4:+.4f} (k=3 has higher inertia — fewer clusters)")

Lab 1 uses k=**4** by design; the elbow suggests k=**3** — business context may still prefer four risk tiers (Lab 6).

---

## 6. Checkpoint summary

In [ ]:
assert len(inertias) == 7
assert suggested_k == 3
inertia_by_k = dict(inertias)
assert abs(inertia_by_k[4] - 45.8634) < 0.1
assert abs(inertia_by_k[3] - 57.1748) < 0.1
print("✓ All checkpoint assertions passed")

---

## Reflection questions

1. Why does inertia always decrease as k increases?
2. Would you always pick the elbow k for a portfolio segmentation product?
3. How does this differ from Day 4 Lab 3 (KNN k sweep with accuracy)?

**Previous:** [Lab 1 — K-Means baseline](lab01_kmeans_baseline.ipynb)  
**Next:** [Lab 3 — DBSCAN clusters](lab03_dbscan_clusters.ipynb)